# Cầu thủ di chuyển bao xa trong một trận?

**Bài toán:** dự đoán tổng quãng đường (km) của một cầu thủ sau trận, từ vị trí, thời gian trên sân và thống kê cấp đội. Dữ liệu World Cup nam 2022. Bạn cần chạy `prepare` trong README một lần trước.

Một dòng = một cầu thủ trong một trận. Bốn vị trí: thủ môn, hậu vệ, tiền vệ, tiền đạo. Đây là hồi quy vì kết quả là số km liên tục.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), Path.cwd() / 'football-distance', Path.cwd().parent] if (p / 'src' / 'project.py').exists()), None)
assert ROOT is not None, 'Không tìm thấy thư mục dự án football-distance'
sys.path.insert(0,str(ROOT / 'src'))
import project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
print('Python:',sys.executable)

## 1. Kiểm tra dữ liệu

Dữ liệu quãng đường đến từ bảng Physical Data của FIFA. OCR đôi khi đọc sai, nên script chỉ nhận cầu thủ có số áo khớp bảng đội hình và ghi lỗi vào báo cáo. Số phút là ước tính từ bảng thay người; có thể lệch do bù giờ.

In [ ]:
df=pd.read_csv(ROOT/'data'/'processed'/'player_matches.csv')
issues=pd.read_csv(ROOT/'reports'/'extraction_issues.csv')
source=pd.read_csv(ROOT/'reports'/'extracted_matches.csv')
print('Trận đọc được:',source.match_id.nunique(),'Cầu thủ hợp lệ:',len(df),'Dòng cần kiểm tra:',len(issues))
display(df.head())
display(df.groupby('position').agg(n=('distance_km','size'),avg_km=('distance_km','mean')).round(2))
display(issues.reason.value_counts().head(10))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for position,group in df.groupby('position'):
    axes[0].hist(group.distance_km,bins=15,alpha=.4,label=position)
axes[0].legend();axes[0].set(xlabel='Distance (km)',ylabel='Count')
axes[1].scatter(df.minutes_played,df.distance_km,alpha=.3)
axes[1].set(xlabel='Approximate minutes played',ylabel='Distance (km)')
plt.tight_layout();plt.show()

## 2. Chọn mô hình

Giữ 20% **trận đấu** để kiểm tra; các cầu thủ cùng trận không bị phân sang train và test. Trên train, GroupKFold so sánh baseline, Ridge và Random Forest bằng MAE. Pipeline xử lý giá trị thiếu và mã hóa vị trí trong từng fold.

In [ ]:
metrics=project.train()
display(pd.DataFrame([metrics]))
display(pd.read_csv(ROOT/'reports'/'model_comparison.csv'))

## 3. Đánh giá theo vị trí

MAE/RMSE tính bằng km; nhỏ hơn là tốt hơn. So sánh với baseline và đọc MAE cùng số mẫu của từng vị trí. Kết quả chỉ phản ánh các trận World Cup 2022 trong tập test.

In [ ]:
display(pd.read_csv(ROOT/'reports'/'position_metrics.csv'))
display(Image(filename=str(ROOT/'reports'/'evaluation.png')))
display(pd.read_csv(ROOT/'reports'/'test_predictions.csv').head(10))

## 4. Dự đoán thử

Ví dụ JSON nhập thông tin sau trận. Trường `warnings` báo đặc điểm ngoài khoảng train; đừng coi dự đoán là số đo thực tế.

In [ ]:
import json
example=json.loads((ROOT/'examples'/'player.json').read_text(encoding='utf-8'))
display(example)
display(project.predict(example))

## 5. Tự luyện

1. So sánh MAE của bốn vị trí và nêu vị trí có ít mẫu.
2. Thử cầu thủ vào sân phút 70: đổi `started=0`, `minutes_played=20`.
3. Giải thích vì sao `distance_km` từ vùng tốc độ không được dùng làm đầu vào.

Nguồn: [FIFA Training Centre](https://www.fifatrainingcentre.com/en/fwc2022/post-match-summaries/post-match-summary-reports.php) và [Fjelstul World Cup Database](https://datahub.io/football/worldcup).